<a href="https://colab.research.google.com/github/Nawat1124/wind-turbine-scada-anomaly-detection/blob/main/04a_final_if_vs_lstm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# 04a Final experiment: IF vs LSTM-AE

This notebook answers **RQ1** only.

Goal:
1. load the fixed model-ready window data;
2. train the final small LSTM-AE once;
3. train the final Isolation Forest once;
4. evaluate both with **global q90** and CARE event-level metrics;
5. save the raw LSTM row scores for the threshold-comparison notebook.



In [ ]:
# =========================================================
# 1. Load data and basic packages
# =========================================================

import os
import random
import joblib
import numpy as np
import pandas as pd

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    pass

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest

# Change this path if your project folder is different.
FREEZE_DIR = "/content/drive/MyDrive/windfarmB_reconstructed_observed_db_v2"

WINDOW_FILE = os.path.join(FREEZE_DIR, "model_ready_simple", "windows_4h_stride2h_bundle.pkl")
EVENT_INFO_FILE = os.path.join(FREEZE_DIR, "event_info_v6.csv")
ROW_LINEAGE_FILE = os.path.join(FREEZE_DIR, "row_lineage_map.parquet")

RESULT_DIR = os.path.join(FREEZE_DIR, "results", "04_final_no_fl")
os.makedirs(RESULT_DIR, exist_ok=True)

bundle = joblib.load(WINDOW_FILE)

df_model = bundle["df_model"].copy().reset_index(drop=True)
KMEANS_COLS = list(bundle["KMEANS_COLS"])

X_train = bundle["X_train"]
X_val = bundle["X_val"]
train_window_meta = bundle["train_window_meta"].reset_index(drop=True)
val_window_meta = bundle["val_window_meta"].reset_index(drop=True)
X_pred_by_event = bundle["X_pred_by_event"]
pred_meta_by_event = {int(k): v.reset_index(drop=True) for k, v in bundle["pred_meta_by_event"].items()}

WINDOW = int(bundle["window"])
STRIDE = int(bundle["stride"])

RANDOM_STATE = 42
Q = 0.90
NORMAL_STATUS_IDS = [0, 2]
CRITICALITY_THRESHOLD = 72
CARE_BETA = 0.5

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("DEVICE:", DEVICE)
print("df_model:", df_model.shape)
print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("WINDOW:", WINDOW, "STRIDE:", STRIDE)
print("Prediction events:", sorted(X_pred_by_event.keys()))

assert X_train.ndim == 3
assert X_val.ndim == 3
assert X_train.shape[1] == WINDOW
assert X_val.shape[1] == WINDOW
assert X_train.shape[2] == len(KMEANS_COLS)


Mounted at /content/drive
DEVICE: cpu
df_model: (560060, 71)
X_train: (30591, 24, 49)
X_val: (3378, 24, 49)
WINDOW: 24 STRIDE: 12
Prediction events: [np.int64(2), np.int64(7), np.int64(19), np.int64(21), np.int64(23), np.int64(27), np.int64(34), np.int64(52), np.int64(53), np.int64(74), np.int64(77), np.int64(82), np.int64(83), np.int64(86), np.int64(87)]


In [ ]:
# =========================================================
# 2. Build official CARE event rows
# =========================================================

# event_info_v6.csv can be semicolon-separated.
event_info = pd.read_csv(EVENT_INFO_FILE, sep=";")
if len(event_info.columns) == 1:
    event_info = pd.read_csv(EVENT_INFO_FILE)

row_lineage_map = pd.read_parquet(ROW_LINEAGE_FILE)

event_info["event_id"] = event_info["event_id"].astype(int)
event_info["asset_id"] = event_info["asset_id"].astype(int)
event_info["event_start_id"] = event_info["event_start_id"].astype(int)
event_info["event_end_id"] = event_info["event_end_id"].astype(int)
event_info["event_label"] = event_info["event_label"].astype(str).str.lower()

row_lineage_map["asset_id"] = row_lineage_map["asset_id"].astype(int)
row_lineage_map["source_session_id"] = row_lineage_map["source_session_id"].astype(str)
row_lineage_map["source_pos"] = row_lineage_map["source_pos"].astype(int)
row_lineage_map["physical_order_index"] = row_lineage_map["physical_order_index"].astype(int)
row_lineage_map["status_type_id"] = row_lineage_map["status_type_id"].astype(int)

parts = []

for _, ev in event_info.iterrows():
    event_id = int(ev["event_id"])
    asset_id = int(ev["asset_id"])
    source_file = f"{event_id}.csv"

    d = row_lineage_map[
        (row_lineage_map["asset_id"] == asset_id)
        & (row_lineage_map["source_session_id"] == source_file)
        & (row_lineage_map["source_pos"] >= int(ev["event_start_id"]))
        & (row_lineage_map["source_pos"] <= int(ev["event_end_id"]))
    ].copy()

    d["event_id"] = event_id
    d["event_label"] = ev["event_label"]
    d["event_description"] = ev.get("event_description", "")

    parts.append(d[[
        "event_id",
        "asset_id",
        "physical_order_index",
        "reconstructed_time",
        "status_type_id",
        "event_label",
        "event_description",
    ]])

care_event_frame = pd.concat(parts, ignore_index=True)
care_event_frame = (
    care_event_frame
    .drop_duplicates(["event_id", "asset_id", "physical_order_index"])
    .sort_values(["event_id", "physical_order_index"])
    .reset_index(drop=True)
)

print("care_event_frame:", care_event_frame.shape)
print(care_event_frame.groupby(["event_id", "event_label"]).size())


care_event_frame: (61616, 7)
event_id  event_label
2         normal          1927
7         anomaly         4465
19        anomaly         2881
21        normal          1297
23        normal          1407
27        anomaly         8785
34        anomaly         3169
52        normal          2017
53        anomaly         6048
74        normal          1921
77        anomaly         8641
82        normal          1729
83        normal         13105
86        normal          1919
87        normal          2305
dtype: int64


In [ ]:
# =========================================================
# 3. Small CARE helper functions
# =========================================================

def f_beta(tp, fp, fn, beta=0.5):
    beta2 = beta ** 2
    denom = (1 + beta2) * tp + beta2 * fn + fp
    if denom == 0:
        return 0.0
    return ((1 + beta2) * tp) / denom


def max_criticality(status, alarm):
    c = 0
    max_c = 0

    for s, a in zip(status, alarm):
        if int(s) in NORMAL_STATUS_IDS:
            if int(a) == 1:
                c += 1
            else:
                c = max(c - 1, 0)
        max_c = max(max_c, c)

    return max_c


def weighted_earliness(alarm):
    alarm = np.asarray(alarm).astype(int)
    n = len(alarm)
    if n == 0:
        return np.nan

    pos = np.linspace(0, 1, n)
    weights = np.where(pos <= 0.5, 1.0, 2.0 * (1.0 - pos))
    return float((weights * alarm).sum() / weights.sum())


def care_evaluate(row_table, alarm_col, model_name):
    pred = row_table[["event_id", "asset_id", "physical_order_index", alarm_col]].copy()
    pred[alarm_col] = pred[alarm_col].fillna(0).astype(int)

    pred = (
        pred
        .groupby(["event_id", "asset_id", "physical_order_index"], as_index=False)
        .agg(alarm=(alarm_col, "max"))
    )

    eval_rows = care_event_frame.merge(
        pred,
        on=["event_id", "asset_id", "physical_order_index"],
        how="left"
    )
    eval_rows["alarm"] = eval_rows["alarm"].fillna(0).astype(int)

    event_rows = []

    for event_id, d in eval_rows.groupby("event_id"):
        d = d.sort_values("physical_order_index")
        label = d["event_label"].iloc[0]
        asset_id = int(d["asset_id"].iloc[0])

        # CARE evaluates alarm activity on normal operating rows.
        normal_rows = d[d["status_type_id"].isin(NORMAL_STATUS_IDS)]
        alarm = normal_rows["alarm"].astype(int).to_numpy()
        n = len(alarm)
        n_alarm = int(alarm.sum())

        coverage = np.nan
        accuracy = np.nan
        earliness = np.nan

        if label == "anomaly":
            coverage = f_beta(tp=n_alarm, fp=0, fn=n - n_alarm, beta=CARE_BETA)
            earliness = weighted_earliness(alarm)
        else:
            accuracy = (n - n_alarm) / n if n > 0 else np.nan

        max_c = max_criticality(
            d["status_type_id"].to_numpy(),
            d["alarm"].to_numpy()
        )

        event_rows.append({
            "model": model_name,
            "event_id": int(event_id),
            "asset_id": asset_id,
            "event_label": label,
            "event_true_label": int(label == "anomaly"),
            "event_pred_label": int(max_c > CRITICALITY_THRESHOLD),
            "n_normal_status_rows": n,
            "n_alarm_rows": n_alarm,
            "alarm_rate_normal_status": n_alarm / n if n > 0 else np.nan,
            "coverage": coverage,
            "accuracy": accuracy,
            "weighted_earliness": earliness,
            "max_criticality": max_c,
            "event_description": d["event_description"].iloc[0],
        })

    detail = pd.DataFrame(event_rows)
    anom = detail[detail["event_label"] == "anomaly"]
    norm = detail[detail["event_label"] == "normal"]

    tp_event = int(((detail["event_true_label"] == 1) & (detail["event_pred_label"] == 1)).sum())
    fn_event = int(((detail["event_true_label"] == 1) & (detail["event_pred_label"] == 0)).sum())
    fp_event = int(((detail["event_true_label"] == 0) & (detail["event_pred_label"] == 1)).sum())

    reliability = f_beta(tp_event, fp_event, fn_event, beta=CARE_BETA)

    summary = pd.DataFrame([{
        "model": model_name,
        "TP/FN/FP": f"{tp_event}/{fn_event}/{fp_event}",
        "mean_coverage": anom["coverage"].mean(),
        "mean_earliness": anom["weighted_earliness"].mean(),
        "event_reliability": reliability,
        "mean_normal_accuracy": norm["accuracy"].mean(),
        "anomaly_alarm_rate": anom["alarm_rate_normal_status"].mean(),
        "normal_alarm_rate": norm["alarm_rate_normal_status"].mean(),
    }])

    summary["CARE_score"] = (
        summary["mean_coverage"]
        + summary["mean_earliness"]
        + summary["event_reliability"]
        + 2 * summary["mean_normal_accuracy"]
    ) / 5

    if detail["event_pred_label"].sum() == 0:
        summary["CARE_score"] = 0.0

    return detail, summary, eval_rows


In [ ]:
# =========================================================
# 4. Train final small LSTM-AE
# =========================================================

n_train, T, d = X_train.shape

lstm_scaler = StandardScaler()
lstm_scaler.fit(X_train.reshape(-1, d))


def transform_3d(X, fitted_scaler):
    n, T_local, d_local = X.shape
    X2 = X.reshape(-1, d_local)
    X2 = fitted_scaler.transform(X2)
    return X2.reshape(n, T_local, d_local).astype(np.float32)


X_train_s = transform_3d(X_train, lstm_scaler)
X_val_s = transform_3d(X_val, lstm_scaler)

train_loader = DataLoader(
    TensorDataset(torch.from_numpy(X_train_s)),
    batch_size=128,
    shuffle=True,
    drop_last=False,
)

val_loader = DataLoader(
    TensorDataset(torch.from_numpy(X_val_s)),
    batch_size=256,
    shuffle=False,
    drop_last=False,
)


class LSTMAutoencoder(nn.Module):
    def __init__(self, n_features, hidden_size=32, latent_size=16):
        super().__init__()
        self.encoder = nn.LSTM(n_features, hidden_size, batch_first=True)
        self.to_latent = nn.Linear(hidden_size, latent_size)
        self.from_latent = nn.Linear(latent_size, hidden_size)
        self.decoder = nn.LSTM(latent_size, hidden_size, batch_first=True)
        self.output_layer = nn.Linear(hidden_size, n_features)

    def forward(self, x):
        _, (h, _) = self.encoder(x)
        z = self.to_latent(h[-1])
        z_seq = z.unsqueeze(1).repeat(1, x.shape[1], 1)
        h0 = self.from_latent(z).unsqueeze(0)
        c0 = torch.zeros_like(h0)
        dec, _ = self.decoder(z_seq, (h0, c0))
        return self.output_layer(dec)


model = LSTMAutoencoder(n_features=d, hidden_size=32, latent_size=16).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()

best_state = None
best_val_loss = np.inf
bad_epochs = 0
history = []

for epoch in range(1, 51):
    model.train()
    train_losses = []

    for (xb,) in train_loader:
        xb = xb.to(DEVICE)
        optimizer.zero_grad()
        recon = model(xb)
        loss = criterion(recon, xb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        train_losses.append(loss.item() * len(xb))

    model.eval()
    val_losses = []

    with torch.no_grad():
        for (xb,) in val_loader:
            xb = xb.to(DEVICE)
            recon = model(xb)
            loss = criterion(recon, xb)
            val_losses.append(loss.item() * len(xb))

    train_loss = np.sum(train_losses) / len(train_loader.dataset)
    val_loss = np.sum(val_losses) / len(val_loader.dataset)

    history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss})
    print(f"Epoch {epoch:03d} | train_loss={train_loss:.6f} | val_loss={val_loss:.6f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        bad_epochs = 0
    else:
        bad_epochs += 1

    if bad_epochs >= 8:
        print("Early stopping.")
        break

model.load_state_dict(best_state)

pd.DataFrame(history).to_csv(os.path.join(RESULT_DIR, "04a_lstm_training_history.csv"), index=False)
torch.save(model.state_dict(), os.path.join(RESULT_DIR, "04a_lstm_small_32_16.pt"))
joblib.dump(lstm_scaler, os.path.join(RESULT_DIR, "04a_lstm_scaler.joblib"))

print("Best val loss:", best_val_loss)


Epoch 001 | train_loss=0.551533 | val_loss=0.334094
Epoch 002 | train_loss=0.308165 | val_loss=0.242656
Epoch 003 | train_loss=0.242670 | val_loss=0.207437
Epoch 004 | train_loss=0.210230 | val_loss=0.178436
Epoch 005 | train_loss=0.186794 | val_loss=0.156061
Epoch 006 | train_loss=0.166633 | val_loss=0.137745
Epoch 007 | train_loss=0.151052 | val_loss=0.125781
Epoch 008 | train_loss=0.139978 | val_loss=0.117606
Epoch 009 | train_loss=0.131520 | val_loss=0.112299
Epoch 010 | train_loss=0.124846 | val_loss=0.108111
Epoch 011 | train_loss=0.119781 | val_loss=0.104764
Epoch 012 | train_loss=0.115431 | val_loss=0.101819
Epoch 013 | train_loss=0.111484 | val_loss=0.098342
Epoch 014 | train_loss=0.107993 | val_loss=0.096632
Epoch 015 | train_loss=0.104966 | val_loss=0.094520
Epoch 016 | train_loss=0.102382 | val_loss=0.092352
Epoch 017 | train_loss=0.099837 | val_loss=0.091123
Epoch 018 | train_loss=0.097739 | val_loss=0.089830
Epoch 019 | train_loss=0.095788 | val_loss=0.087983
Epoch 020 | 

In [ ]:
# =========================================================
# 5. LSTM validation and prediction row scores
# =========================================================

@torch.no_grad()
def lstm_scores(X_scaled):
    model.eval()
    loader = DataLoader(
        TensorDataset(torch.from_numpy(X_scaled.astype(np.float32))),
        batch_size=256,
        shuffle=False,
    )

    window_scores = []
    row_scores = []

    for (xb,) in loader:
        xb = xb.to(DEVICE)
        recon = model(xb)
        err = (xb - recon) ** 2
        window_scores.append(err.mean(dim=(1, 2)).cpu().numpy())
        row_scores.append(err.mean(dim=2).cpu().numpy())

    return np.concatenate(window_scores), np.concatenate(row_scores)


lstm_val_window_score, lstm_val_row_score = lstm_scores(X_val_s)

# Global q90 is calculated from flattened validation timestep-level row errors.
lstm_global_q90 = float(np.quantile(lstm_val_row_score.reshape(-1), Q))
print("LSTM global q90:", lstm_global_q90)

# Save validation row scores with asset/state metadata for the threshold notebook.
val_records = []

for w in range(len(val_window_meta)):
    end_row = int(val_window_meta.loc[w, "end_row"])
    start_row = int(val_window_meta.loc[w, "start_row"]) if "start_row" in val_window_meta.columns else end_row - T + 1
    row_ids = np.arange(start_row, end_row + 1)

    for j, df_row in enumerate(row_ids):
        val_records.append({
            "df_model_row": int(df_row),
            "asset_id": int(df_model.loc[df_row, "asset_id"]),
            "kmeans_state": int(df_model.loc[df_row, "kmeans_state"]),
            "lstm_row_score": float(lstm_val_row_score[w, j]),
        })

lstm_val_flat = pd.DataFrame(val_records)
lstm_val_flat.to_csv(os.path.join(RESULT_DIR, "04a_lstm_validation_flat_row_scores.csv"), index=False)

# Prediction: convert window-timestep scores back to one score per reconstructed row.
pred_tables = []

for event_id in sorted(X_pred_by_event):
    Xp_s = transform_3d(X_pred_by_event[event_id], lstm_scaler)
    _, pred_row_score = lstm_scores(Xp_s)
    meta = pred_meta_by_event[event_id]

    records = []

    for w in range(len(meta)):
        end_row = int(meta.loc[w, "end_row"])
        start_row = int(meta.loc[w, "start_row"]) if "start_row" in meta.columns else end_row - T + 1
        row_ids = np.arange(start_row, end_row + 1)

        for j, df_row in enumerate(row_ids):
            records.append({
                "df_model_row": int(df_row),
                "lstm_row_score": float(pred_row_score[w, j]),
            })

    row_pred = pd.DataFrame(records)

    # If one row is covered by multiple overlapping windows, use max aggregation.
    row_pred = (
        row_pred
        .groupby("df_model_row", as_index=False)
        .agg(
            lstm_row_score=("lstm_row_score", "max"),
            n_covering_windows=("lstm_row_score", "size"),
        )
    )

    base = df_model.iloc[row_pred["df_model_row"].to_numpy()].copy().reset_index(drop=True)
    base["df_model_row"] = row_pred["df_model_row"].to_numpy()
    base["event_id"] = int(event_id)
    base["lstm_row_score"] = row_pred["lstm_row_score"].to_numpy()
    base["n_covering_windows"] = row_pred["n_covering_windows"].to_numpy()

    pred_tables.append(base)

lstm_rows = pd.concat(pred_tables, ignore_index=True)
lstm_rows["lstm_alarm_global_q90"] = (lstm_rows["lstm_row_score"] > lstm_global_q90).astype(int)

lstm_rows.to_csv(os.path.join(RESULT_DIR, "04a_lstm_prediction_row_scores_raw_and_global_q90.csv"), index=False)

print("lstm_val_flat:", lstm_val_flat.shape)
print("lstm_rows:", lstm_rows.shape)
print("LSTM global alarm rate:", lstm_rows["lstm_alarm_global_q90"].mean())


LSTM global q90: 0.14210481941699982
lstm_val_flat: (81072, 4)
lstm_rows: (72096, 76)
LSTM global alarm rate: 0.25969540612516645


In [ ]:
# =========================================================
# 6. CARE evaluation for LSTM global q90
# =========================================================

lstm_detail, lstm_summary, lstm_eval_rows = care_evaluate(
    row_table=lstm_rows,
    alarm_col="lstm_alarm_global_q90",
    model_name="LSTM-AE global q90",
)

print("LSTM global q90 summary")
display(lstm_summary)

print("LSTM event detail")
display(lstm_detail.sort_values("event_id"))

lstm_detail.to_csv(os.path.join(RESULT_DIR, "04a_lstm_global_q90_event_detail.csv"), index=False)
lstm_summary.to_csv(os.path.join(RESULT_DIR, "04a_lstm_global_q90_summary.csv"), index=False)


LSTM global q90 summary


,model,TP/FN/FP,mean_coverage,mean_earliness,event_reliability,mean_normal_accuracy,anomaly_alarm_rate,normal_alarm_rate,CARE_score
0,LSTM-AE global q90,4/2/1,0.569469,0.21523,0.769231,0.819823,0.217315,0.180177,0.638715


LSTM event detail


,model,event_id,asset_id,event_label,event_true_label,event_pred_label,n_normal_status_rows,n_alarm_rows,alarm_rate_normal_status,coverage,accuracy,weighted_earliness,max_criticality,event_description
0,LSTM-AE global q90,2,13,normal,0,0,1872,162,0.086538,NaN,0.913462,NaN,17,NaN
1,LSTM-AE global q90,7,13,anomaly,1,1,4210,840,0.199525,0.554822,NaN,0.198495,141,high temperature in transformer cell
2,LSTM-AE global q90,19,11,anomaly,1,1,2855,789,0.276357,0.656297,NaN,0.278518,128,high temperature in transformer cell
3,LSTM-AE global q90,21,0,normal,0,0,1269,291,0.229314,NaN,0.770686,NaN,33,NaN
4,LSTM-AE global q90,23,6,normal,0,0,1353,200,0.147820,NaN,0.852180,NaN,20,NaN
5,LSTM-AE global q90,27,7,anomaly,1,0,8404,1559,0.185507,0.532445,NaN,0.200053,62,Turbine is stopped due to a main bearing damage
6,LSTM-AE global q90,34,14,anomaly,1,0,3046,368,0.120814,0.407260,NaN,0.112245,32,high temperature in transformer cell
7,LSTM-AE global q90,52,14,normal,0,0,2012,296,0.147117,NaN,0.852883,NaN,16,NaN
8,LSTM-AE global q90,53,6,anomaly,1,1,5711,1220,0.213623,0.575961,NaN,0.232760,118,Rotor Bearing 2 - Damage
9,LSTM-AE global q90,74,11,normal,0,0,1867,392,0.209963,NaN,0.790037,NaN,34,NaN


In [ ]:
# =========================================================
# 7. Train Isolation Forest baseline
# =========================================================

if_scaler = StandardScaler()
if_scaler.fit(X_train.reshape(-1, d))

X_train_if_s = transform_3d(X_train, if_scaler)
X_val_if_s = transform_3d(X_val, if_scaler)


def make_if_features(X):
    # One 4-hour window becomes one fixed-length vector.
    win_mean = X.mean(axis=1)
    win_std = X.std(axis=1)
    win_range = X.max(axis=1) - X.min(axis=1)

    t = np.arange(X.shape[1], dtype=np.float32)
    t = t - t.mean()
    win_slope = np.einsum("ntd,t->nd", X, t) / np.sum(t ** 2)

    return np.concatenate([win_mean, win_std, win_range, win_slope], axis=1).astype(np.float32)


X_if_train = make_if_features(X_train_if_s)
X_if_val = make_if_features(X_val_if_s)

iso = IsolationForest(
    n_estimators=300,
    max_samples="auto",
    contamination="auto",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
iso.fit(X_if_train)

if_val_score = -iso.score_samples(X_if_val)

# Same q90 logic as LSTM. Repeating the window score over rows does not change the quantile.
if_global_q90 = float(np.quantile(np.repeat(if_val_score, T), Q))
print("IF global q90:", if_global_q90)

if_pred_tables = []

for event_id in sorted(X_pred_by_event):
    Xp_s = transform_3d(X_pred_by_event[event_id], if_scaler)
    Xp_if = make_if_features(Xp_s)
    window_score = -iso.score_samples(Xp_if)
    meta = pred_meta_by_event[event_id]

    records = []

    for w in range(len(meta)):
        end_row = int(meta.loc[w, "end_row"])
        start_row = int(meta.loc[w, "start_row"]) if "start_row" in meta.columns else end_row - T + 1
        row_ids = np.arange(start_row, end_row + 1)

        for df_row in row_ids:
            records.append({
                "df_model_row": int(df_row),
                "if_row_score": float(window_score[w]),
            })

    row_pred = pd.DataFrame(records)
    row_pred = (
        row_pred
        .groupby("df_model_row", as_index=False)
        .agg(
            if_row_score=("if_row_score", "max"),
            n_covering_windows=("if_row_score", "size"),
        )
    )

    base = df_model.iloc[row_pred["df_model_row"].to_numpy()].copy().reset_index(drop=True)
    base["df_model_row"] = row_pred["df_model_row"].to_numpy()
    base["event_id"] = int(event_id)
    base["if_row_score"] = row_pred["if_row_score"].to_numpy()
    base["n_covering_windows"] = row_pred["n_covering_windows"].to_numpy()

    if_pred_tables.append(base)

if_rows = pd.concat(if_pred_tables, ignore_index=True)
if_rows["if_alarm_global_q90"] = (if_rows["if_row_score"] > if_global_q90).astype(int)

if_rows.to_csv(os.path.join(RESULT_DIR, "04a_if_prediction_row_scores_global_q90.csv"), index=False)
joblib.dump(if_scaler, os.path.join(RESULT_DIR, "04a_if_scaler.joblib"))
joblib.dump(iso, os.path.join(RESULT_DIR, "04a_isolation_forest.joblib"))

print("if_rows:", if_rows.shape)
print("IF global alarm rate:", if_rows["if_alarm_global_q90"].mean())


IF global q90: 0.48255096576389467
if_rows: (72096, 76)
IF global alarm rate: 0.25466045272969373


In [ ]:
# =========================================================
# 8. CARE evaluation for IF global q90
# =========================================================

if_detail, if_summary, if_eval_rows = care_evaluate(
    row_table=if_rows,
    alarm_col="if_alarm_global_q90",
    model_name="IF global q90",
)

print("IF global q90 summary")
display(if_summary)

print("IF event detail")
display(if_detail.sort_values("event_id"))

if_detail.to_csv(os.path.join(RESULT_DIR, "04a_if_global_q90_event_detail.csv"), index=False)
if_summary.to_csv(os.path.join(RESULT_DIR, "04a_if_global_q90_summary.csv"), index=False)


IF global q90 summary


,model,TP/FN/FP,mean_coverage,mean_earliness,event_reliability,mean_normal_accuracy,anomaly_alarm_rate,normal_alarm_rate,CARE_score
0,IF global q90,6/0/5,0.580139,0.225996,0.6,0.780465,0.226019,0.219535,0.593413


IF event detail


,model,event_id,asset_id,event_label,event_true_label,event_pred_label,n_normal_status_rows,n_alarm_rows,alarm_rate_normal_status,coverage,accuracy,weighted_earliness,max_criticality,event_description
0,IF global q90,2,13,normal,0,0,1872,304,0.162393,NaN,0.837607,NaN,36,NaN
1,IF global q90,7,13,anomaly,1,1,4210,670,0.159145,0.486212,NaN,0.143016,83,high temperature in transformer cell
2,IF global q90,19,11,anomaly,1,1,2855,803,0.281261,0.661777,NaN,0.300602,156,high temperature in transformer cell
3,IF global q90,21,0,normal,0,1,1269,371,0.292356,NaN,0.707644,NaN,92,NaN
4,IF global q90,23,6,normal,0,0,1353,213,0.157428,NaN,0.842572,NaN,36,NaN
5,IF global q90,27,7,anomaly,1,1,8404,1372,0.163256,0.493809,NaN,0.177260,162,Turbine is stopped due to a main bearing damage
6,IF global q90,34,14,anomaly,1,1,3046,475,0.155942,0.480186,NaN,0.156841,102,high temperature in transformer cell
7,IF global q90,52,14,normal,0,0,2012,312,0.155070,NaN,0.844930,NaN,60,NaN
8,IF global q90,53,6,anomaly,1,1,5711,1603,0.280686,0.661140,NaN,0.289366,191,Rotor Bearing 2 - Damage
9,IF global q90,74,11,normal,0,1,1867,452,0.242100,NaN,0.757900,NaN,112,NaN


In [ ]:
# =========================================================
# 9. Final RQ1 table
# =========================================================

rq1_table = pd.concat([lstm_summary, if_summary], ignore_index=True)

rq1_table = rq1_table[[
    "model",
    "TP/FN/FP",
    "mean_coverage",
    "mean_earliness",
    "mean_normal_accuracy",
    "anomaly_alarm_rate",
    "normal_alarm_rate",
    "CARE_score",
]]

print("RQ1: LSTM-AE vs Isolation Forest under global q90")
display(rq1_table)

rq1_table.to_csv(os.path.join(RESULT_DIR, "04a_RQ1_lstm_vs_if_global_q90.csv"), index=False)
print("Saved to:", RESULT_DIR)


RQ1: LSTM-AE vs Isolation Forest under global q90


,model,TP/FN/FP,mean_coverage,mean_earliness,mean_normal_accuracy,anomaly_alarm_rate,normal_alarm_rate,CARE_score
0,LSTM-AE global q90,4/2/1,0.569469,0.215230,0.819823,0.217315,0.180177,0.638715
1,IF global q90,6/0/5,0.580139,0.225996,0.780465,0.226019,0.219535,0.593413


Saved to: /content/drive/MyDrive/windfarmB_reconstructed_observed_db_v2/results/04_final_no_fl
